# Snowflake & Tableau Ossie Quickstart

Convert a Tableau workbook `.twb` connected to the Snowflake TPCH sample dataset to [Apache Ossie](https://github.com/apache/ossie) format, convert it to a native Snowflake semantic view, and generate a Tableau Data Source `.tds` that connects live to the Snowflake semantic view. This process is designed to simluate how to use the Apache Ossie spec to "shift left" and migrate Tableau semantics to Snowflake.

Optional dbt Cloud Semantic Layer step queries the same Snowflake sample tables through MetricFlow, and the optional Power BI step exports the TPCH tables from Snowflake into a Fabric lakehouse and uses the Microsoft Ossie converter to facilitate PowerBI Direct Lake mode.

```mermaid
flowchart LR
  sf["<b>Snowflake</b><br/><i>SNOWFLAKE_SAMPLE_DATA.</i><br/><i>TPCH_SF1</i>"] --> sv["<b>Snowflake</b><br/>Semantic view<br/><i>sql/</i><br/><i>create_semantic_view.sql</i>"]
  twb["<b>Tableau</b><br/><i>workbooks/tpch_snowflake.twb</i>"] --> yaml["<b>Apache Ossie</b><br/><i>out/</i><br/><i>tpch_snowflake__</i><br/><i>tpch_semantic_model.ossie.yaml</i>"]
  yaml --> sv
  sv --> tds["<b>Tableau</b><br/><i>out/</i><br/><i>tpch_snowflake__</i><br/><i>tpch_semantic_model.tds</i>"]
  subgraph optionalLive ["Optional live"]
    dbt["<b>dbt Labs</b><br/>dbt Cloud Semantic Layer<br/><i>dbt_sl.py</i>, <i>dbt_tpch/</i>"]
    pbi["<b>Microsoft</b><br/>Power BI Direct Lake<br/><i>pbi_directlake.py</i>"]
  end
  yaml --> dbt
  sf --> dbt
  sf --> csvExport["<b>Snowflake</b><br/>Table export<br/><i>pbi_directlake.py</i>"]
  csvExport --> lh["<b>Microsoft Fabric</b><br/>Lakehouse <i>TPCH_SF1</i>"]
  yaml --> pbiYaml["<b>Apache Ossie</b><br/><i>out/*.pbi.ossie.yaml</i>"]
  pbiYaml --> bim["<b>Microsoft</b><br/><i>out/model.bim</i>"]
  lh --> bim
  bim --> pbi
  classDef optional fill:#f59e0b33,stroke:#f59e0b,stroke-width:2px,stroke-dasharray:5 3
  class dbt,pbi,csvExport,lh,pbiYaml,bim optional
  style optionalLive fill:#f59e0b11,stroke:#f59e0b,stroke-dasharray:5 3
```

## 1. Setup

Clone [Apache Ossie](https://github.com/apache/ossie), put this folder at the clone root, then run:

```sh
jupyter notebook tableau-ossie-quickstart/tableau_ossie_quickstart.ipynb
```

### Prerequisites

**Required (main path)**

- This folder (detected by its `helpers/connect.py` + `README.md` signature, not by name) at the root of
  an [Apache Ossie](https://github.com/apache/ossie) clone, with a `converters/tableau` (or `tableau`)
  checkout providing `ossie_tableau`. That converter isn't in `apache/ossie` yet — it's in the separate
  [apache/ossie-converters](https://github.com/apache/ossie-converters) repo; see this project's
  [README](README.md#prerequisites) for the exact pinned-commit clone commands, and re-check that pin
  before relying on it once the converter lands upstream.
- Python 3.11+ Jupyter kernel
- Tableau Desktop, to open [tpch_snowflake.twb](workbooks/tpch_snowflake.twb)
- A Snowflake account with `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`. 
- Snowflake CLI with a `[default]` entry in `~/.snowflake/connections.toml`.

- Python packages installed in the kernel **before you run the notebook**:
  - `snowflake-connector-python`: Snowflake connection (`helpers/connect.py`)
  - `PyYAML>=6.0`, `pydantic>=2.0`, `sqlglot>=30.12.0`: Ossie models and SQL translation
  - `pyarrow>=21`: Parquet export for Power BI (needed for `store_decimal_as_integer`)
  - `metricflow`: in-process dbt MSI conversion

  ```sh
  pip install snowflake-connector-python "PyYAML>=6.0" "sqlglot>=30.12.0" "pydantic>=2.0" "pyarrow>=21" metricflow
  # uv-created venv without pip:
  # uv pip install --python <path to kernel python> snowflake-connector-python "PyYAML>=6.0" "sqlglot>=30.12.0" "pydantic>=2.0" "pyarrow>=21" metricflow
  ```

- **One-time Snowflake setup, as `ACCOUNTADMIN`.**
  - Edit the three `SET` lines at the top of [sql/setup_snowflake.sql](sql/setup_snowflake.sql) (your user, warehouse, and the dbt Cloud service role). Then run it in a Snowsight worksheet, or from the terminal:

    ```sh
    snow sql -c <connection> --role ACCOUNTADMIN -f sql/setup_snowflake.sql
    ```

  - It creates `OSSIE_QS_ROLE`, the database `OSSIE_TABLEAU`, the schemas `OSSIE_QS` and `OSSIE_QS_DBT`, and the dbt role's grants. It is idempotent.
  - To start over, run [sql/teardown_snowflake.sql](sql/teardown_snowflake.sql) and then the setup script again:

    ```sh
    snow sql -c <connection> --role ACCOUNTADMIN -f sql/teardown_snowflake.sql
    ```

### Optional: dbt Cloud and Power BI (section 7)

Snowflake credentials come only from the `[default]` entry in `~/.snowflake/connections.toml`; the notebook sets `OSSIE_QS_ROLE` itself. `.env` is used only for the two optional steps below. To enable either, copy `.env.example` to `.env` in this folder and fill it in.

```sh
cp .env.example .env
```

**dbt Cloud Semantic Layer**

- A dbt Cloud account with a dedicated Project `ossie-qs`
- A **deployment** environment targeting database `OSSIE_TABLEAU`, schema `OSSIE_QS_DBT`, with Semantic Layer credentials.
- `.env`: set the environment variables `DBT_CLOUD_URL`, `DBT_CLOUD_TOKEN`, `DBT_CLOUD_SNOWFLAKE_USER`, `DBT_CLOUD_SNOWFLAKE_ROLE`.
- `metricflow` comes from the package list above; the MSI cell falls back to `uv run --with metricflow` if it is missing.

**Power BI**

- Service principal with Fabric workspace Contributor and `Lakehouse.ReadWrite.All`; `converters/microsoft` present in the Ossie clone.
- `.env`: set the environment variables `PBI_TENANT_ID`, `PBI_CLIENT_ID`, `PBI_CLIENT_SECRET`, `PBI_WORKSPACE_ID`. With these blank, section 7 checks the DAX locally and skips Fabric.
- Optional: `OSSIE_QS_TPCH_FILTER=YYYY-MM` exports one `L_SHIPDATE` month instead of full SF1.

In [ ]:
# Step 1: Locate the Ossie clone and make its converter packages importable.
# Ossie = Apache Ossie, an open semantic-model format; Tableau/dbt/Power BI converters live in its repo.
from __future__ import annotations

import dataclasses
import json
import os
import re
import sys
from pathlib import Path

# Notebook can be launched from the Ossie clone root or from the quickstart folder itself.
cwd = Path.cwd().resolve()


# Converters ship under converters/tableau or tableau depending on the checkout layout.
def tableau_src(root: Path) -> Path | None:
    for rel in (root / "converters" / "tableau" / "src", root / "tableau" / "src"):
        if (rel / "ossie_tableau").is_dir():
            return rel
    return None


# Every converter src dir that exists gets added to sys.path below.
def src_dirs(root: Path) -> list[Path]:
    found = []
    for rel in (
        root / "python" / "src",
        root / "ossie" / "python" / "src",
        root / "converters" / "tableau" / "src",
        root / "tableau" / "src",
        root / "converters" / "dbt" / "src",
        root / "dbt" / "src",
        root / "converters" / "microsoft" / "src",
        root / "microsoft" / "src",
    ):
        if rel.is_dir():
            found.append(rel)
    return found


# A quickstart folder is identified by its signature files, not by name: this folder
# ships under whatever name its own repo uses (e.g. "quickstart" or "tableau-ossie-quickstart").
def is_quickstart_dir(p: Path) -> bool:
    return (p / "helpers" / "connect.py").is_file() and (p / "README.md").is_file()


# Resolve `here` (the Ossie clone root, or wrapper repo root) and `QS` (the quickstart folder)
# from wherever the notebook was started, regardless of what the quickstart folder is named.
if is_quickstart_dir(cwd):
    here, QS = cwd.parent, cwd
else:
    sub = next((p for p in cwd.iterdir() if p.is_dir() and is_quickstart_dir(p)), None)
    if sub is None:
        raise SystemExit(f"run from the Ossie clone root or the quickstart folder; cwd={cwd}")
    here, QS = cwd, sub

# OSSIE = Ossie clone root; fail early if neither layout matches.
if tableau_src(here):
    OSSIE = here
elif tableau_src(here / "ossie"):
    OSSIE = here / "ossie"
else:
    raise SystemExit(
        "put the quickstart folder at the root of an Apache Ossie clone "
        "(https://github.com/apache/ossie) or this wrapper repo; "
        f"missing converters/tableau or tableau under {here}"
    )

# OUT collects generated artifacts (YAML, loss.json, .tds, manifests); SQL holds the Snowflake scripts.
OUT = QS / "out"
SQL = QS / "sql"
OUT.mkdir(parents=True, exist_ok=True)
# Put quickstart/helpers + converter sources first on sys.path so `connect` and `ossie_tableau` import.
for rel in [QS / "helpers", *src_dirs(OSSIE)]:
    path = str(rel)
    if path not in sys.path:
        sys.path.insert(0, path)
print("ossie", OSSIE.relative_to(here) if OSSIE != here else ".")
print("quickstart", QS.relative_to(here))
print("python", Path(sys.executable).name)

# Role for every Snowflake session below; sql/setup_snowflake.sql (run once as ACCOUNTADMIN) creates it.
import importlib
import connect as _connect_mod

_connect_mod = importlib.reload(_connect_mod)  # the kernel may hold an older copy without DEFAULT_ROLE
connect = _connect_mod.connect
ROLE = _connect_mod.DEFAULT_ROLE

Step 1 checks your setup and queries `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1` with the first statement in [sql/metrics_snowflake.sql](sql/metrics_snowflake.sql). 

Known live values:
- **total_revenue = 218102223885.0001**
- **revenue_per_customer = 2181109.4832293302**


In [ ]:
# Step 1: Check the Snowflake setup, then compute the baseline metrics from the TPCH sample data.
# connect and ROLE come from the setup cell.

# Schema every later step reads from or writes to.
SCHEMA = "OSSIE_TABLEAU.OSSIE_QS"
# Golden values for TPCH SF1: total_revenue and revenue_per_customer.
# Every downstream path (Snowflake semantic view, dbt, Power BI) is asserted against these.
EXPECTED = 218102223885.0001
EXPECTED_RPC = 2181109.4832293302

# Execute a .sql file statement by statement (the connector runs one statement per execute).
# only=N runs just the Nth statement (0-based); the cursor then holds that statement's result.
def run_sql_file(cur, path: Path, only: int | None = None) -> None:
    # Strip -- comments before splitting on semicolons so notes cannot leak SQL.
    lines = [ln for ln in path.read_text().splitlines() if not ln.strip().startswith("--")]
    stmts = [s.strip() for s in "\n".join(lines).split(";") if s.strip()]
    for stmt in (stmts if only is None else [stmts[only]]):
        cur.execute(stmt)

# connect() lives in helpers/connect.py (the [default] connections.toml entry); the role is explicit.
conn = connect(role=ROLE)
cur = conn.cursor()
try:
    cur.execute(f"USE ROLE {ROLE}")
    print("role", ROLE)
    # Pre-flight: privileged objects come from sql/setup_snowflake.sql, run once as ACCOUNTADMIN.
    SETUP_HINT = "run sql/setup_snowflake.sql as ACCOUNTADMIN, then rerun this cell"
    try:
        cur.execute("SHOW GRANTS ON SCHEMA OSSIE_TABLEAU.OSSIE_QS_DBT")
        grants = cur.fetchall()
    except Exception as exc:
        raise SystemExit(f"OSSIE_TABLEAU is not set up for {ROLE} ({exc}); {SETUP_HINT}") from exc
    # Optional: the dbt Cloud role must be able to write to OSSIE_QS_DBT.
    from dbt_sl import load_dbt_env  # reads .env as well as the process environment

    dbt_role = load_dbt_env(QS).get("DBT_CLOUD_SNOWFLAKE_ROLE", "").strip()
    if dbt_role:
        # SHOW GRANTS columns: privilege is row[1], grantee_name is row[5].
        privs = {row[1] for row in grants if str(row[5]).upper() == dbt_role.upper()}
        if not {"CREATE TABLE", "CREATE VIEW"} <= privs:
            raise SystemExit(f"{dbt_role} lacks CREATE on OSSIE_QS_DBT (has {sorted(privs)}); {SETUP_HINT}")
        print("dbt role", dbt_role, "can create in OSSIE_QS_DBT")
    else:
        print("skipped dbt role check (DBT_CLOUD_SNOWFLAKE_ROLE not set)")
    # Run the raw-table metrics query (statement 0 of metrics_snowflake.sql).
    run_sql_file(cur, SQL / "metrics_snowflake.sql", only=0)
    revenue, rpc = cur.fetchone()
    print("queried SNOWFLAKE_SAMPLE_DATA.TPCH_SF1")
    print("total_revenue", revenue)
    print("revenue_per_customer", rpc)
    # Fail fast if the data or SQL drifts from the golden values.
    assert abs(float(revenue) - EXPECTED) < 1.0, revenue
    assert abs(float(rpc) - EXPECTED_RPC) < 0.01, rpc
finally:
    # Always release the connection, even if an assert fails.
    conn.close()


## 2. Open the Tableau workbook (Optional)

Open [tpch_snowflake.twb](workbooks/tpch_snowflake.twb) in Tableau Desktop. Note the Ossie converter reads the Tableau XML and does not need a live connection; this step is so you can see the source model.

The sheet already has the TPC-H tables and measures `total_revenue` / `revenue_per_customer`.

![Tableau sheet](img/tab1.png)

The stored Snowflake connection is the host `account.snowflakecomputing.com`. Data → Edit Connection: replace Server with your account host and authenticate.

![Snowflake key-pair connection](img/tab2.png)

Data Source: database `SNOWFLAKE_SAMPLE_DATA`, schema `TPCH_SF1`. The model is `LINEITEM` to `ORDERS` / `PART` / `SUPPLIER`, `ORDERS` to `CUSTOMER` to `NATION`.

![TPC-H data source](img/tab3.png)


## 3. Workbook to Ossie

`ossie-tableau convert` reads [tpch_snowflake.twb](workbooks/tpch_snowflake.twb) as XML. It writes Ossie YAML plus a classified `.loss.json` under `quickstart/out/`. The stem is `<workbook>__<datasource>` (`tpch_snowflake__tpch_semantic_model`). The following cell is the in-process equivalent to:

```sh
uv run --project converters/tableau ossie-tableau convert \
  quickstart/workbooks/tpch_snowflake.twb -o quickstart/out
```


In [ ]:
# Step 3: Convert the Tableau workbook (.twb) into an Ossie semantic model.
# convert_file returns one result per datasource: an Ossie document plus a list of "losses" (Tableau features with no Ossie equivalent).
from ossie_tableau.convert import convert_file

TWB = QS / "workbooks" / "tpch_snowflake.twb"
# Parse the workbook XML and map its datasource, joins, and calculations into Ossie.
results = convert_file(TWB)
assert len(results) == 1 and results[0].document is not None
r01 = results[0]
# safe_stem turns the datasource caption into a filesystem-safe name for output files.
def safe_stem(s: str) -> str:
    return re.sub(r"[^0-9A-Za-z_]+", "_", s).strip("_") or "x"

stem01 = f"{TWB.stem}__{safe_stem(r01.datasource)}"
yaml01 = OUT / f"{stem01}.ossie.yaml"
# Write the portable Ossie YAML plus a loss.json recording anything that could not convert.
yaml01.write_text(r01.document.to_ossie_yaml())
(OUT / f"{stem01}.loss.json").write_text(
    json.dumps([dataclasses.asdict(x) for x in r01.losses], indent=2)
)
print(f"OK   {stem01}: {r01.stats} losses={len(r01.losses)}")
# Expected shape of the TPCH workbook: 6 tables, 5 joins, 2 metrics.
assert stem01 == "tpch_snowflake__tpch_semantic_model"
assert r01.stats.get("datasets") == 6
assert r01.stats.get("relationships") == 5
assert r01.stats.get("metrics") == 2
# All six datasets must point at the Snowflake sample TPCH tables.
assert {d.source for d in r01.document.datasets} == {
    f"SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.{t}"
    for t in ("LINEITEM", "ORDERS", "CUSTOMER", "NATION", "PART", "SUPPLIER")
}
# Revenue-per-customer uses a Tableau LOD (level-of-detail) calc, so a "lod" loss entry is expected.
assert any(x.kind == "lod" for x in r01.losses)


### Reading the convert output

```
OK   tpch_snowflake__tpch_semantic_model: {'datasets': 6, 'fields': 54, 'relationships': 5, 'metrics': 2, 'unconverted_calcs': 1, 'row_level_fields': 1} losses=3
```

- **`OK <stem>`**: the output file stem, `<workbook>__<datasource>` (here the `tpch_semantic_model` datasource). The `.ossie.yaml` and `.loss.json` are written under that name in `out/`.
- **`datasets: 6`**: `LINEITEM`, `ORDERS`, `CUSTOMER`, `NATION`, `PART`, `SUPPLIER`.
- **`fields: 54`**: columns carried over as Ossie fields, plus the row-level `customer_label`.
- **`relationships: 5`**: the TPC-H joins.
- **`metrics: 2`**: `total_revenue` and `revenue_per_customer`.
- **`unconverted_calcs: 1`**: the FIXED LOD `customer_lifetime_revenue` is parked (kind `lod`). It sums an expression across `LINEITEM` and `CUSTOMER`, which convert doesn't translate. A simpler single-table `{FIXED [d]: SUM([x])}` is emitted with window SQL instead (kind `lod_translated`).
- **`row_level_fields: 1`**: `customer_label`.
- **`losses=3`**: one warning (`lod`) plus two info losses (`relationship_cardinality`, `semantic_values`). Nothing is dropped silently.

Print the Tableau and Snowflake dialects for `total_revenue` and `revenue_per_customer`.


In [ ]:
# Step 3: Inspect the metric expressions Ossie produced.
# Each metric stores one expression per dialect (TABLEAU formula and SNOWFLAKE SQL); this flattens them to {dialect: expression}.
def dialect_map(metric) -> dict[str, str]:
    out = {}
    for d in metric.expression.dialects:
        out[str(getattr(d.dialect, "value", d.dialect))] = d.expression
    return out

# Index every metric in the converted document by name.
metrics = {m.name: dialect_map(m) for m in r01.document.metrics}
for name in ("total_revenue", "revenue_per_customer"):
    assert name in metrics, sorted(metrics)
    print(name)
    for dialect, expr in metrics[name].items():
        print(f"  {dialect}: {expr}")

# Tableau dialect keeps Tableau syntax (COUNTD); Snowflake dialect is plain SQL a semantic view can run.
assert "L_EXTENDEDPRICE" in metrics["total_revenue"]["TABLEAU"]
assert "L_DISCOUNT" in metrics["total_revenue"]["SNOWFLAKE"]
assert "COUNTD" in metrics["revenue_per_customer"]["TABLEAU"]
# Summarize the categories of loss recorded for this workbook.
print("loss.json kinds", sorted({x.kind for x in r01.losses}))


## 4. Snowflake semantic view

Native [`CREATE SEMANTIC VIEW`](https://docs.snowflake.com/en/sql-reference/sql/create-semantic-view) from [sql/create_semantic_view.sql](sql/create_semantic_view.sql). The view lives in `OSSIE_TABLEAU.OSSIE_QS` and reads `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`.

Creates `SV_01_TPCH`, queries `total_revenue` and `revenue_per_customer`. Known live values:   
- **total_revenue = 218102223885.0001**
- **revenue_per_customer = 2181109.4832293302**

> **Note:** Do not use [`SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML`](https://docs.snowflake.com/en/sql-reference/stored-procedures/system_create_semantic_view_from_ossie_yaml). Converter writes Ossie `0.2.0.dev0`. As of 2026-10-27, Snowflake accepts `0.1.1` only there is no Ossie 2.0.0 support yet. `0.2.0.dev0` is not compatible until Snowflake lists that version and accepts these metric expressions.

In [ ]:
# Step 4: Create a native Snowflake semantic view from the checked-in DDL and query its metrics.
# Fully qualified name of the semantic view (reused by the TDS export in step 5).
VIEW = f"{SCHEMA}.SV_01_TPCH"
ddl = (SQL / "create_semantic_view.sql").read_text()
# Preview: confirm the DDL targets the sample data, then show the CREATE line and the two metric definitions.
assert "SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM" in ddl
print(next(l for l in ddl.splitlines() if l.strip().startswith("CREATE OR REPLACE")))
print("  ...")
print(next(l for l in ddl.splitlines() if "total_revenue" in l))
print(next(l for l in ddl.splitlines() if l.strip().startswith("revenue_per_customer")))

conn = connect(role=ROLE)
cur = conn.cursor()
try:
    cur.execute(f"USE ROLE {ROLE}")
    # Create (or replace) the semantic view in Snowflake.
    run_sql_file(cur, SQL / "create_semantic_view.sql")
    # Statement 1 of metrics_snowflake.sql: SEMANTIC_VIEW(...) evaluates the metrics through the view's joins.
    run_sql_file(cur, SQL / "metrics_snowflake.sql", only=1)
    total, rpc = cur.fetchone()
    print("total_revenue", total)
    print("revenue_per_customer", rpc)
    # Same golden values as the raw SQL baseline: the semantic view must agree.
    assert abs(float(total) - EXPECTED) < 1.0, total
    assert abs(float(rpc) - EXPECTED_RPC) < 0.01, rpc
finally:
    conn.close()
    # The view is intentionally kept; step 5 exports it and step 8 (cleanup) drops it.
    print("kept", VIEW)

## 5. Snowflake semantic view to Tableau TDS

[`SYSTEM$EXPORT_TDS_FROM_SEMANTIC_VIEW`](https://docs.snowflake.com/en/sql-reference/functions/system_export_tds_from_semantic_view) writes a `.tds` that points at `SV_01_TPCH` as one flat table. Metrics use `AGG()` and there are no Tableau relationships/joins as these have been implemented in the Snowflake semantic view definition.

Equivalent to:

```sql
SELECT SYSTEM$EXPORT_TDS_FROM_SEMANTIC_VIEW('OSSIE_TABLEAU.OSSIE_QS.SV_01_TPCH');
```

In [ ]:
# Step 5: Export the semantic view as a Tableau Data Source (.tds) so Tableau can connect to it directly.
import xml.etree.ElementTree as ET

conn = connect(role=ROLE)
cur = conn.cursor()
try:
    cur.execute(f"USE ROLE {ROLE}")
    # Snowflake generates the TDS XML for the view server-side.
    cur.execute(f"select SYSTEM$EXPORT_TDS_FROM_SEMANTIC_VIEW('{VIEW}')")
    xml = cur.fetchall()[0][0]
finally:
    conn.close()

# Save the .tds under out/ so it can be opened or published from Tableau.
tds_path = OUT / f"{stem01}.tds"
tds_path.write_text(xml)
# Count <relationship> elements; the export is expected to contain none.
rels = len(list(ET.fromstring(xml).iter("relationship")))
print(f"OK   {stem01}.tds bytes={len(xml)} relationships={rels}")
# Sanity checks: valid XML header, points at our view, includes the revenue metric.
assert xml.lstrip().startswith("<?xml"), xml[:80]
assert "SV_01_TPCH" in xml
assert "TOTAL_REVENUE" in xml
# AGG(%1) is Tableau's placeholder for "aggregate the field using the view's metric definition".
assert "AGG(%1)" in xml
assert rels == 0
# Security check: the file must not embed passwords, tokens, or secrets.
for attr in ("password", "token", "secret"):
    assert not re.search(rf'\s{attr}="[^"]+"', xml), attr
print("view", VIEW)



### Reading the TDS output

- **`OK <stem>.tds`**: `SYSTEM$EXPORT_TDS_FROM_SEMANTIC_VIEW` wrote the `.tds` to `out/` under the same stem as the Ossie YAML.
- **`relationships=0`**: the export is one flat table over `SV_01_TPCH`. There are no Tableau joins. Metrics use `AGG()` so Tableau pushes them to the Snowflake semantic view.
- **`view`**: the fully qualified name passed to the function. The cell also asserts that the XML starts with `<?xml`, contains `SV_01_TPCH` and `TOTAL_REVENUE` (Snowflake uppercases the metric; the visible caption is `Total Revenue`), wraps it in `AGG(%1)`, and writes no `password`, `token` or `secret` attribute. Connection details come from the live session.


## 6. What convert records as a loss

[lod.twb](workbooks/lod.twb) includes two viz-dependent LODs on `LINEITEM`: `{INCLUDE [L_SHIPMODE]: SUM([L_EXTENDEDPRICE])}` and `{EXCLUDE [L_RETURNFLAG]: SUM([L_QUANTITY])}`. 

Convert keeps both as metrics and attaches window SQL, then writes a non-empty `.loss.json`. Kind `sql_unreconciled` means the SQL is table-wide `OVER`, not Tableau Compute Using. 

![lod.twb INCLUDE EXCLUDE LOD](img/tab-lod.png)


In [ ]:
# Step 6: Convert a workbook with INCLUDE/EXCLUDE LOD (level-of-detail) calculations and review what is recorded as a loss.
# LOD = a Tableau calc that aggregates at a different grain than the view (e.g. add or remove a dimension).
LOD_TWB = QS / "workbooks" / "lod.twb"
# Same converter as step 3, different input workbook.
lod_results = convert_file(LOD_TWB)
assert len(lod_results) == 1 and lod_results[0].document is not None
rr = lod_results[0]
# Name output files after the workbook and datasource, as in step 3.
stem06 = f"{LOD_TWB.stem}__{safe_stem(rr.datasource)}"
(OUT / f"{stem06}.ossie.yaml").write_text(rr.document.to_ossie_yaml())
loss06 = OUT / f"{stem06}.loss.json"
loss06.write_text(json.dumps([dataclasses.asdict(x) for x in rr.losses], indent=2))
print(f"OK   {stem06}: {rr.stats} losses={len(rr.losses)}")
print("loss.json", loss06.read_text().strip())
# Two LOD calcs are translated to SQL window functions; the converter flags them 'sql_unreconciled'
# (SQL was generated but not proven equivalent to Tableau's result), so review them before relying on them.
assert len(rr.losses) == 2
assert {x.kind for x in rr.losses} == {"sql_unreconciled"}
assert rr.stats["unconverted_calcs"] == 0
# Compare the Tableau INCLUDE formula with its Snowflake SQL translation.
include = dialect_map(next(m for m in rr.document.metrics if m.name == "Sales_Include_Shipmode"))
print("Sales_Include_Shipmode TABLEAU  ", include["TABLEAU"])
print("Sales_Include_Shipmode SNOWFLAKE", include["SNOWFLAKE"])
# Tableau side keeps INCLUDE on L_SHIPMODE; Snowflake side becomes a window (OVER) expression.
assert "INCLUDE" in include["TABLEAU"] and "L_SHIPMODE" in include["TABLEAU"]
assert "OVER" in include["SNOWFLAKE"]


### Reading the LOD output

- **Summary line**: same fields as the first convert. Here `lod.twb` yields 2 metrics, `unconverted_calcs: 0`, and `losses=2`. The calcs are in the model; convert still recorded a loss for each.
- **`losses=2` / non-empty `loss.json`**: both LODs are `sql_unreconciled` (approximated). Convert attached table-wide window SQL and refused to call it equivalent to Tableau Compute Using.
- **`unconverted_calcs: 0`**: the calcs are metrics, not parked. Contrast section 3's parked `{FIXED}` `customer_lifetime_revenue` (`unconverted_calcs: 1`, kind `lod`).
- **`Sales_Include_Shipmode TABLEAU`**: the original `{INCLUDE [LINEITEM].[L_SHIPMODE]: SUM([LINEITEM].[L_EXTENDEDPRICE])}`.
- **`Sales_Include_Shipmode SNOWFLAKE`**: `SUM((L_EXTENDEDPRICE)) OVER (PARTITION BY (L_SHIPMODE))`. The EXCLUDE sibling is `SUM((L_QUANTITY)) OVER ()`. Tableau EXCLUDE means viz grain minus that dimension; convert emits a table-wide window. That SQL is best-effort, not a MetricFlow or DAX translation. Optional section 7 still deploys `total_revenue` / `revenue_per_customer` only.


## 7. dbt Cloud and PBI conversions (Optional)

### dbt Semantic Layer (Snowflake)

The [dbt Semantic Layer](https://docs.getdbt.com/docs/use-dbt-semantic-layer/dbt-sl?version=2) deploys [dbt_tpch/](dbt_tpch/) on project `ossie-qs` and queries `total_revenue` / `revenue_per_customer` on `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`. Requires the environment variables `DBT_CLOUD_URL` and `DBT_CLOUD_TOKEN`. dbt Ossie convertor lives under `converters/dbt` in the Ossie clone.

1. **MetricFlow Semantic Interface (MSI) convert.** Convert the TPC-H Ossie YAML to a [MSI manifest](https://docs.getdbt.com/reference/artifacts/sl-manifest?version=2) so you can see the metric names the converter emitted. Writes `out/semantic_manifest.json`


In [ ]:
# Convert the Ossie model to a dbt Semantic Layer manifest (MetricFlow "MSI").
import subprocess

# Output path for the generated MetricFlow semantic manifest.
dest = OUT / "semantic_manifest.json"

# Metrics can appear as a dict or a list in the manifest; return their names either way.
def _msi_metric_names(data: dict) -> list[str]:
    metrics = data.get("metrics")
    if isinstance(metrics, dict):
        return sorted(metrics)
    if isinstance(metrics, list):
        return sorted(m.get("name") for m in metrics if isinstance(m, dict) and m.get("name"))
    return []

# Find the dbt converter project in the Ossie clone (used by the uv fallback below).
def _dbt_project() -> Path | None:
    for rel in (OSSIE / "converters" / "dbt", OSSIE / "dbt"):
        if (rel / "pyproject.toml").is_file():
            return rel
    return None

# Preferred path: run the converter in-process if ossie_dbt is importable.
try:
    from ossie_dbt.ossie_to_msi import OssieToMSIConverter

    msi = OssieToMSIConverter().convert(r01.document)
    dest.write_text(msi.output.json(by_alias=True, exclude_none=True, indent=2))
    print("Written", dest.relative_to(QS))
except (ModuleNotFoundError, ImportError) as exc:
    # Fallback: run the converter through `uv` in its own environment.
    project = _dbt_project()
    if project is None:
        raise SystemExit(f"skipped dbt MSI ({exc}); no converters/dbt or dbt project under {OSSIE}") from exc
    print("in-process MSI unavailable (", exc, "); using uv --project ", project, sep="")
    subprocess.run(
        ["uv", "run", "--project", str(project), "--with", "metricflow", "ossie-dbt", "ossie-to-msi",
         "-i", str(yaml01), "-o", str(dest)],
        check=True,
    )
    print("Written", dest.relative_to(QS))

data = json.loads(dest.read_text())
names = _msi_metric_names(data)
print("msi metrics", names)
# Both quickstart metrics must make it into the manifest.
assert set(names) >= {"total_revenue", "revenue_per_customer"}


2. **Configure Snowflake for dbt Cloud.** Create a service user (this repo uses `DBT_CLOUD_SVC` and a key pair). Its default role needs `SELECT` permissions on `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`; pass that role as `dbt_role` to [sql/setup_snowflake.sql](sql/setup_snowflake.sql), which grants it `CREATE TABLE` / `CREATE VIEW` in `OSSIE_TABLEAU.OSSIE_QS_DBT`.


3. **dbt Cloud project.** Create a new dbt Cloud project called `ossie-qs` and git-connect this repo with subdirectory `dbt_tpch`.

4. **Create the Environment** Add a Snowflake connection profile and credentials for the dbt Cloud service user created in step 2. Create a deployment environment named `OSSIE_QS_SF`.

![dbt Cloud environment OSSIE_QS_SF](img/dbt_cloud_2.png)

5. **Create the job `OSSIE_QS_SF_PARSE`** in the `OSSIE_QS_SF` environment with these two commands, in order:

1. `dbt run --select stg_* metricflow_time_spine` 

Builds in `OSSIE_TABLEAU.OSSIE_QS_DBT` the three staging views (`stg_customer`, `stg_lineitem`, `stg_orders`) over the TPC-H sample tables and the `metricflow_time_spine` table of daily dates. The semantic models read the staging views, and MetricFlow uses the time spine for time-based queries. This is the only step that writes to Snowflake, which is why the dbt role needs `CREATE TABLE` / `CREATE VIEW` on that schema.

2. `dbt parse` 

Validates the semantic models and metrics (`total_revenue`, `customers`, `revenue_per_customer`) and writes the `semantic_manifest.json` the Semantic Layer reads. No data is moved and metrics are not materialized: each query compiles to SQL over the views at request time.

![dbt Cloud parse job create](img/dbt_cloud_2a.png)

6. Save and run the job and verify all steps have run successfully.

![dbt Cloud parse job success](img/dbt_cloud_3.png)

7. **Studio.** Once 4c succeeds, dbt Cloud Studio shows the MetricFlow metrics (`total_revenue`, `customers`, `revenue_per_customer`) in `models/metrics/tpch.yml`.

![dbt Cloud Studio MetricFlow metrics](img/dbt_cloud.png)


8. **Semantic Layer credentials.** Job credentials and SL credentials are two different objects. A green job run does not mean GraphQL can reach Snowflake vis the dbt Semantic Layer. Edit the `Analytics Credentials` in the `ossie-qs` project and point them at the same Snowflake user, role, warehouse, and key pair from step 2. 

![dbt Cloud project ossie-qs](img/dbt_cloud_4.png)



9. **Notebook step.** Set `DBT_CLOUD_URL`, `DBT_CLOUD_TOKEN` and `DBT_CLOUD_PROJECT=ossie-qs`. `deploy_and_query` triggers the dbt Cloud job and checks `total_revenue` / `revenue_per_customer` over GraphQL.


In [ ]:
# deploy to dbt Cloud and query both metrics through the Semantic Layer.
# Runs only when DBT_CLOUD_URL and DBT_CLOUD_TOKEN are set.
from dbt_sl import cloud_configured, deploy_and_query, load_dbt_env

if not cloud_configured(QS):
    print("skipped dbt Cloud Semantic Layer (set DBT_CLOUD_URL and DBT_CLOUD_TOKEN)")
else:
    # The dbt job fails unless its role has CREATE on OSSIE_QS_DBT (granted by sql/setup_snowflake.sql).
    if not load_dbt_env(QS).get("DBT_CLOUD_SNOWFLAKE_ROLE", "").strip():
        print("warning: DBT_CLOUD_SNOWFLAKE_ROLE not set; the dbt job fails unless its role can create in OSSIE_TABLEAU.OSSIE_QS_DBT (see sql/setup_snowflake.sql)")
    # Deploy the manifest to dbt Cloud, query both metrics, and compare to the golden values.
    values = deploy_and_query(QS)
    revenue = values["total_revenue"]
    rpc = values["revenue_per_customer"]
    assert abs(float(revenue) - EXPECTED) < 1.0, revenue
    assert abs(float(rpc) - EXPECTED_RPC) < 0.01, rpc


10. **GraphQL explorer.** The cell below prints your explorer URL (from `DBT_CLOUD_URL` via `dbt_sl.sl_graphql_url`), the auth header, your deployment environment id and three GraphQL snippets to paste in.

In [ ]:
#print the Semantic Layer GraphQL explorer URL, header and queries for your account.
from dbt_sl import explorer_info

info = explorer_info(QS)
if not info["url"]:
    print("skipped (set DBT_CLOUD_URL and DBT_CLOUD_TOKEN)")
else:
    env_id = info["environment_id"] or "<environmentId>"
    print("Open:   ", info["url"])
    print('Header:  {"Authorization": "Bearer <DBT_CLOUD_TOKEN>"}')
    if info["environment_id"]:
        print(f"Environment: {info['environment_name']} (id {env_id})")
    else:
        print("Environment: not found; copy the id from the dbt Cloud environment URL")
    print(f"""
# 1. list metrics
{{
  metricsPaginated(environmentId: {env_id}, pageNum: 1) {{
    items {{ name type }}
  }}
}}

# 2. run both metrics
mutation {{
  createQuery(
    environmentId: {env_id}
    metrics: [{{name: "total_revenue"}}, {{name: "revenue_per_customer"}}]
  ) {{ queryId }}
}}

# 3. poll with the queryId from step 2 until status is SUCCESSFUL
{{
  query(environmentId: {env_id}, queryId: "<queryId>") {{
    status
    error
    jsonResult(encoded: false)
  }}
}}""")


Open the link above to call the dbt Semantic Layer [GraphQL API](https://docs.getdbt.com/docs/dbt-cloud-apis/sl-graphql).

`metricsPaginated` lists `customers` (SIMPLE), `revenue_per_customer` (RATIO) and `total_revenue` (SIMPLE):

![dbt Semantic Layer metrics](img/dbt2.png)

`createQuery`, then polls `query` until `SUCCESSFUL`. `jsonResult` returns the SF1 metrics (`total_revenue` **218102223885.0001**, `revenue_per_customer` **2181109.4832293303**):

![dbt Semantic Layer query pins](img/dbt1.png)

### Power BI (Fabric Direct Lake)

Snowflake exports the six TPC-H tables into a [Fabric lakehouse](https://learn.microsoft.com/en-us/fabric/data-engineering/lakehouse-overview) (`ossie_qs_lh`, schema `TPCH_SF1`). The `ossie_microsoft` converter builds `out/model.bim` from a sibling `.pbi.ossie.yaml`, and the cell deploys it as the [Direct Lake](https://learn.microsoft.com/en-us/fabric/fundamentals/direct-lake-overview) semantic model `ossie-qs-tpch`. Power BI reads the lakehouse copy, not a live Snowflake connection.

1. **Set `PBI_TENANT_ID`, `PBI_CLIENT_ID`, `PBI_CLIENT_SECRET` and `PBI_WORKSPACE_ID`.** Without them the cell checks the DAX, writes `out/model.bim`, and skips Fabric. `LINEITEM` has 6,001,215 rows; set `OSSIE_QS_TPCH_FILTER=YYYY-MM` for a one-month extract.

2. **Run the cell.** [pbi_directlake.py](helpers/pbi_directlake.py) adds `DISCOUNTED_PRICE` and `ORDER_CUSTKEY` (the customer on each line item) to `LINEITEM`, because the converter only handles `SUM` over one column. It then loads the lakehouse, deploys, and refreshes.

In [ ]:
#Build a Power BI Direct Lake semantic model from the Ossie YAML on Microsoft Fabric,
# query it, and compare the results to the golden values. Skips itself if Fabric settings are missing.
import importlib
import pbi_directlake

# Reload so edits to pbi_directlake.py apply without restarting the kernel.
pbi_directlake = importlib.reload(pbi_directlake)
run = pbi_directlake.run
# Args: Ossie YAML from step 3, quickstart dir, output dir, Ossie clone, and the expected metric values.
run(yaml01, QS, OUT, OSSIE, EXPECTED, EXPECTED_RPC)


### Reading the Power BI output

- **`rewrote out/...pbi.ossie.yaml`**: the sibling YAML, with SQL expressions reformatted for the converter (`SUM(DISCOUNTED_PRICE)`).
- **`lakehouse ossie_qs_lh <id>`**: the lakehouse was found or created by name.
- **`export SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.<TABLE>`**: a table is being pulled from Snowflake. `LINEITEM` is joined to `ORDERS` to add `ORDER_CUSTKEY`, then sent in chunks (default 1,000,000 rows).
- **`UserWarning` lines**: expected. The converter drops Tableau `custom_extensions` and display labels, and infers measure data types from the DAX. Measures with no home table land on `CUSTOMER`.
- **`loaded TPCH_SF1.LINEITEM rows 6001215`**: the table is in the lakehouse. Row counts are a sanity check against SF1.
- **`measures [...]`** and **`wrote out/model.bim`**: the two DAX measures, and the TMSL model with Direct Lake partitions on the lakehouse tables.
- **`skipped Power BI`**: `PBI_*` is unset. `model.bim` was written and the DAX checked; Fabric and Snowflake were not called.
- **`updated ossie-qs-tpch <id>`** and **`refresh DirectLakeFraming completed`**: the semantic model was deployed (or updated in place) and Direct Lake reframed onto the loaded tables.
- **`executeQueries {... PowerBINotAuthorizedException}`** and **`model was not queried`**: the REST query was denied for this identity. The deploy did not fail. Open the printed link and compare `218102223885.0001` / `2181109.4832293302` yourself.
- **`open https://...`**: the deployed semantic model in Fabric.

If Fabric allows [`executeQueries`](https://learn.microsoft.com/en-us/rest/api/power-bi/datasets/execute-queries), it also prints the metrics (`total_revenue=218102223885.0001` / `revenue_per_customer=2181109.4832293302`).

#### What it looks like in Fabric

Open the printed link to reach Power BI semantic model which has been published to Fabric `ossie-qs-tpch`. 

Model view of the converted tables and relationships (note the use of Direct Lake as the storage mode):

![Fabric Direct Lake model](img/fab-model.png)

A report visual (`total_revenue` by `N_NAME` and showing `total_revenue=218102223885.0001` / `revenue_per_customer=2181109.4832293302`) built on the semantic model:

![Fabric report: total_revenue by nation](img/fab-report.png)

In [ ]:
# Step 8: Teardown. Standalone: needs no other cell, so it works after a kernel restart or a partial run.
# Everything below is commented out on purpose because it drops objects. Uncomment to run.
# Drops the Snowflake schemas OSSIE_TABLEAU.OSSIE_QS and OSSIE_QS_DBT and deletes the Fabric semantic model and lakehouse.
# Kept: the database OSSIE_TABLEAU, OSSIE_QS_ROLE and its grants (full reset: sql/teardown_snowflake.sql as ACCOUNTADMIN),
# the dbt Cloud project, environment and job (remove those in the dbt UI), and the local files in out/.
# #
# import importlib
# import sys
# from pathlib import Path

# # Find quickstart/ from wherever the notebook was started (Ossie clone root or quickstart/ itself).
# cwd = Path.cwd().resolve()
# QS = next((p for p in (cwd, cwd / "quickstart") if (p / "helpers" / "connect.py").is_file()), None)
# if QS is None:
#     raise SystemExit(f"run from the Ossie clone root or quickstart/; cwd={cwd}")
# OSSIE = QS.parent / "ossie" if (QS.parent / "ossie").is_dir() else QS.parent
# if str(QS / "helpers") not in sys.path:
#     sys.path.insert(0, str(QS / "helpers"))

# from connect import DEFAULT_ROLE, connect
# from snowflake.connector.errors import ProgrammingError

# failures = []


# # Each drop is independent: a missing database or missing privilege is reported, not fatal.
# def drop_schema(cur, schema: str) -> None:
#     try:
#         cur.execute(f"DROP SCHEMA IF EXISTS {schema} CASCADE")
#         print("dropped", schema)
#     except ProgrammingError as exc:
#         if exc.errno == 2003:  # the database does not exist (setup never ran), so there is nothing to drop
#             print("nothing to drop:", schema)
#         elif getattr(exc, "sqlstate", None) == "42501":  # insufficient privileges: another role owns it
#             print("skipped", schema, "(role lacks OWNERSHIP)")
#         else:
#             failures.append(exc)
#             print("failed", schema, exc)


# try:
#     conn = connect(role=DEFAULT_ROLE)
# except Exception as exc:  # role or connection missing: setup never ran or no [default] entry
#     print("skipped Snowflake cleanup:", type(exc).__name__, exc)
# else:
#     try:
#         cur = conn.cursor()
#         for schema in ("OSSIE_TABLEAU.OSSIE_QS", "OSSIE_TABLEAU.OSSIE_QS_DBT"):
#             drop_schema(cur, schema)
#     except Exception as exc:
#         failures.append(exc)
#         print("failed Snowflake cleanup:", type(exc).__name__, exc)
#     finally:
#         conn.close()

# # Fabric: no-ops with a message when PBI_* is unset or the items do not exist.
# try:
#     import pbi_directlake

#     pbi_directlake = importlib.reload(pbi_directlake)  # the kernel may hold an older copy
#     pbi_directlake.delete_fabric_objects(QS, OSSIE)
# except Exception as exc:
#     print("skipped Fabric cleanup:", type(exc).__name__, exc)

# if failures:
#     raise failures[0]


## Done

You've now taken the TPC-H sample data all the way around the loop.

You queried it live in Snowflake, turned the Tableau workbook into Ossie YAML, created a Snowflake semantic view, and wrote a `.tds` from `SYSTEM$EXPORT_TDS_FROM_SEMANTIC_VIEW`.

The LOD example showed convert writing a non-empty `.loss.json`: `{INCLUDE}` / `{EXCLUDE}` stay metrics (`unconverted_calcs: 0`) and still record `sql_unreconciled` because the window SQL is table-wide, not Tableau Compute Using. The main convert's `lod` loss is the parked FIXED calculation.

If you ran the optional section, you also deployed the TPC-H metrics to the dbt Cloud Semantic Layer and published the model to Power BI as Direct Lake on a Fabric export of those tables. One thing to know: the Power BI numbers come from that export, not from a live Snowflake DirectQuery connection.
